# Full Data Pipeline Overview

This notebook implements a robust data pipeline to extract, process, and aggregate export trade data for all EU27 country pairs.

## Pipeline stages

| Stage | Description | Output folder |
|---|---|---|
| Main data pull | Monthly SITC-level intra-EU export flows via Eurostat Comext (DS-059331) | `main_data/` |
| Node features | Per-country monthly features (GDP, ESI, HICP, PPI, multilateral resistance, bond yields) | `node_features/` |
| Edge features | Directed bilateral structural features (CEPII GeoDist), time-expanded | `edge_features/` |
| Graph features | Euro-area-wide features broadcast to all nodes (EURIBOR 3M, VSTOXX) | `graph_features/` |
| GNN assembly | Stack all sources into tensors ready for graph neural network training | `eu_data_pipeline/` |

## Time range
`2014-01` → `2025-11` (Croatia joined mid-2013; starting from 2014-01 gives a stable 27-node network)

## Prerequisites before running
- **`dist_cepii.xls`** — download from http://www.cepii.fr/distance/dist_cepii.zip and place in the notebook directory (or update `GEODIST_FILE` in Cell B)
- **`VSTOXX50.txt`** — download the historical VSTOXX file from stoxx.com and place in the notebook directory (or update `VSTOXX_FILE` in Cell B)

## Resume logic
Every section checks whether its output file already exists before running. The main Comext pull also has fine-grained per-(reporter, month) resume via a progress JSON file. Re-running any cell is safe.


In [ ]:
import os
import sys
import json
import time
import threading
import itertools
from io import StringIO
from datetime import datetime
from pathlib import Path
from itertools import product as iterproduct

import requests
import pandas as pd
import numpy as np

print("Imports loaded.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell B — All configuration variables
# ══════════════════════════════════════════════════════════════════════════════

# ── Time range ────────────────────────────────────────────────────────────────
START_MONTH = "2014-01"   # Croatia joined mid-2013; 2014-01 gives a fixed 27-node network
END_MONTH   = "2025-11"   # Script skips months not yet published by Eurostat

# ── Country lists ─────────────────────────────────────────────────────────────
# The Eurostat Comext API (DS-059331) uses GR for Greece; all other APIs and
# all output files use the standard Eurostat code EL.
COMEXT_COUNTRIES = [
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES",
    "FI", "FR", "GR", "HR", "HU", "IE", "IT", "LT", "LU",
    "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK",
]  # NOTE: GR — used ONLY in the Comext pull. Do not use elsewhere.

COUNTRIES = [
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL",
    "ES", "FI", "FR", "HR", "HU", "IE", "IT", "LT", "LU",
    "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK",
]  # EL: Greece — canonical list used everywhere outside the Comext pull.

# EU27 country codes
# AT=Austria    BE=Belgium    BG=Bulgaria   CY=Cyprus       CZ=Czechia    DE=Germany
# DK=Denmark    EE=Estonia    EL=Greece     ES=Spain        FI=Finland    FR=France
# HR=Croatia    HU=Hungary    IE=Ireland    IT=Italy        LT=Lithuania  LU=Luxembourg
# LV=Latvia     MT=Malta      NL=Netherlands PL=Poland      PT=Portugal   RO=Romania
# SE=Sweden     SI=Slovenia   SK=Slovakia

# ── Derived time index ────────────────────────────────────────────────────────
MONTHLY_IDX = pd.date_range(START_MONTH, END_MONTH, freq="MS")

# ── Output folders ────────────────────────────────────────────────────────────
DATA_DIR      = Path("eu_data_pipeline")
MAIN_DATA_DIR = DATA_DIR / "main_data"
NODE_DIR      = DATA_DIR / "node_features"
EDGE_DIR      = DATA_DIR / "edge_features"
GRAPH_DIR     = DATA_DIR / "graph_features"

for _d in [MAIN_DATA_DIR, NODE_DIR, EDGE_DIR, GRAPH_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Derived file paths ────────────────────────────────────────────────────────
RAW_TRADE_CSV     = MAIN_DATA_DIR / "eu_intra_trade_sitc.csv"
IMPUTED_TRADE_CSV = MAIN_DATA_DIR / "eu_intra_trade_sitc_imputed.csv"
PROGRESS_FILE     = MAIN_DATA_DIR / "eu_intra_trade_progress.json"
GEODIST_FILE      = Path("dist_cepii.xls")   # place in notebook directory, or update path
VSTOXX_FILE       = Path("VSTOXX50.txt")     # place in notebook directory, or update path

# ── API endpoints ─────────────────────────────────────────────────────────────
COMEXT_BASE = "https://ec.europa.eu/eurostat/api/comext/dissemination/statistics/1.0/data/DS-059331"
ESTAT_BASE  = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
ECB_BASE    = "https://data-api.ecb.europa.eu/service/data"

# ── Main data pull settings ───────────────────────────────────────────────────
FLOW        = "2"          # 2 = exports (1 = imports)
INDICATOR   = "VALUE_EUR"
SLEEP_SEC   = 1.0          # seconds between API calls — increase if rate-limited
MAX_RETRIES = 3

PRODUCT_GROUPS = {
    "TOTAL":               ["TOTAL"],
    "Food_drinks_tobacco": ["0", "1"],
    "Raw_materials":       ["2", "4"],
    "Energy":              ["3"],
    "Chemicals":           ["5"],
    "Machinery_vehicles":  ["7"],
    "Other_manufactured":  ["6", "8"],
}
CATS = [k for k in PRODUCT_GROUPS if k != "TOTAL"]   # product columns excluding the aggregate

# ── Imputation settings ───────────────────────────────────────────────────────
ZERO_THRESHOLD = 0.015   # category shares below this → imputed as 0
MIN_HISTORY    = 3       # minimum prior observations required for pair/reporter fallback

# ── Node feature settings ─────────────────────────────────────────────────────
HICP_UNIT = "I15"   # Index, 2015 = 100
HICP_AGGREGATES = {
    "FOOD":           "hicp_food_monthly.csv",
    "NRG":            "hicp_nrg_monthly.csv",
    "IGD_NNRG":       "hicp_igd_nnrg_monthly.csv",
    "TOT_X_NRG_FOOD": "hicp_tot_x_nrg_food_monthly.csv",
}

PPI_UNIT  = "I21"       # Index, 2021 = 100
PPI_INDIC = "PRC_PRR"   # Producer prices, domestic market

PPI_DIVISIONS = {
    "Raw_materials":       ["B08"],
    "Energy":              ["C19", "D35"],
    "Food_drinks_tobacco": ["C10", "C11", "C12"],
    "Chemicals":           ["C20", "C21", "C22", "C23"],
    "Machinery_vehicles":  ["C26", "C27", "C28", "C29", "C30"],
    "Other_manufactured":  ["C13", "C14", "C15", "C16", "C17",
                            "C18", "C24", "C25", "C31", "C32"],
}

# ── Edge feature settings ─────────────────────────────────────────────────────
ISO3_TO_EUROSTAT = {
    "AUT": "AT", "BEL": "BE", "BGR": "BG", "CYP": "CY", "CZE": "CZ",
    "DEU": "DE", "DNK": "DK", "EST": "EE", "GRC": "EL", "ESP": "ES",
    "FIN": "FI", "FRA": "FR", "HRV": "HR", "HUN": "HU", "IRL": "IE",
    "ITA": "IT", "LTU": "LT", "LUX": "LU", "LVA": "LV", "MLT": "MT",
    "NLD": "NL", "POL": "PL", "PRT": "PT", "ROM": "RO", "SWE": "SE",
    "SVN": "SI", "SVK": "SK",
}

CEPII_COLS = [
    "comlang_ethno",  # ethnic/demographic language overlap (≥9% population)
    "contig",         # shared land border
    "smctry",         # were/are the same country at some point
    "dist",           # distance between most-populated cities (km)
    "distcap",        # capital-to-capital distance (km)
    "distw",          # population-weighted distance (km)
    "distwces",       # CES-weighted population distance used in gravity models (km)
]

# ── Graph feature settings ────────────────────────────────────────────────────
ECB_EURIBOR_SERIES = "FM/M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA"

print(f"Monthly index: {MONTHLY_IDX[0].date()} → {MONTHLY_IDX[-1].date()}  ({len(MONTHLY_IDX)} periods)")
print(f"Countries ({len(COUNTRIES)}): {COUNTRIES}")
print(f"Output root: {DATA_DIR.resolve()}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell C — Shared utility functions
# Every function used by more than one section lives here.
# ══════════════════════════════════════════════════════════════════════════════

def build_months(start_month: str, end_month: str) -> list:
    """Generate YYYY-MM strings from start_month to end_month inclusive."""
    months, current = [], datetime.strptime(start_month, "%Y-%m")
    end = datetime.strptime(end_month, "%Y-%m")
    while current <= end:
        months.append(current.strftime("%Y-%m"))
        current = current.replace(month=current.month % 12 + 1,
                                  year=current.year + (current.month == 12))
    return months


def already_done(name: str, out_dir: Path = NODE_DIR) -> bool:
    """Return True if the named file already exists in out_dir."""
    return (out_dir / name).exists()


def save(df: pd.DataFrame, name: str, out_dir: Path = NODE_DIR) -> None:
    """Save df to out_dir/name and print a one-line summary."""
    path = out_dir / name
    df.to_csv(path)
    nans = df.isna().sum().sum()
    print(f"  saved  {path}  shape={df.shape}  NaNs={nans}")


def broadcast_graph_feature(
    series: pd.Series,
    countries: list = COUNTRIES,
    monthly_idx: pd.DatetimeIndex = MONTHLY_IDX,
) -> pd.DataFrame:
    """Broadcast a scalar monthly series to a (T × N) DataFrame (same value every country)."""
    series = series.reindex(monthly_idx)
    return pd.DataFrame(
        np.tile(series.values.reshape(-1, 1), (1, len(countries))),
        index=monthly_idx,
        columns=countries,
    )


def json_stat_to_df(js: dict, value_col: str = "value") -> pd.DataFrame:
    """Convert a Eurostat JSON:stat response into a long-format DataFrame."""
    if "value" not in js:
        raise RuntimeError(f"No 'value' key in response. Keys: {list(js.keys())}")
    if not js["value"]:
        raise RuntimeError("Empty 'value' payload — no data for these filters.")
    dims       = js["dimension"]
    dim_keys   = list(js["id"])
    categories = [list(dims[k]["category"]["index"].keys()) for k in dim_keys]
    idx        = pd.MultiIndex.from_product(categories, names=dim_keys)
    if isinstance(js["value"], dict):
        dense = pd.Series(np.nan, index=range(len(idx)))
        for k, v in js["value"].items():
            dense.iloc[int(k)] = v
        values = dense
    else:
        values = pd.Series(js["value"], dtype=float)
    return pd.DataFrame({value_col: values.values}, index=idx).reset_index()


def _fetch_one(dataset_id: str, param_tuples: list, retries: int = 3) -> dict:
    """Single HTTP request to Eurostat with spinner and retry logic."""
    url  = f"{ESTAT_BASE}/{dataset_id}"
    done = threading.Event()

    def _spinner():
        for elapsed in itertools.count(1):
            if done.is_set():
                break
            sys.stdout.write(f"\r    waiting ... {elapsed}s")
            sys.stdout.flush()
            time.sleep(1)
        sys.stdout.write("\r" + " " * 30 + "\r")
        sys.stdout.flush()

    for attempt in range(retries):
        t = threading.Thread(target=_spinner, daemon=True)
        t.start()
        try:
            r = requests.get(url, params=param_tuples, timeout=60)
            done.set(); t.join()
            r.raise_for_status()
            js = r.json()
            if "error" in js:
                err = js["error"]
                if isinstance(err, list): err = err[0]
                raise RuntimeError(
                    f"Eurostat API error {err.get('status','?')}: "
                    f"{err.get('label','unknown error')}"
                )
            if "warning" in js:
                label = js["warning"].get("label", "")
                if "ASYNCHRONOUS" in label.upper():
                    raise RuntimeError("Unexpected async response.")
            return js
        except RuntimeError:
            done.set(); t.join()
            raise
        except Exception as exc:
            done.set(); t.join()
            done.clear()
            print(f"  [attempt {attempt+1}/{retries}] {exc}")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"Failed after {retries} attempts")


def eurostat_json(dataset_id: str, params: dict) -> pd.DataFrame:
    """Fetch a full Eurostat dataset one country at a time to avoid async issues."""
    countries   = params.get("geo", [])
    if not isinstance(countries, list):
        countries = [countries]
    base_tuples = [(k, v) for k, v in params.items() if k != "geo"]
    all_long    = []

    for i, country in enumerate(countries):
        sys.stdout.write(f"\r  [{i+1:2d}/{len(countries)}] fetching {country} ...        ")
        sys.stdout.flush()
        try:
            js   = _fetch_one(dataset_id, base_tuples + [("geo", country)])
            long = json_stat_to_df(js)
            all_long.append(long)
        except RuntimeError as exc:
            print(f"\n  WARNING: {country} skipped — {exc}")
        time.sleep(0.5)

    sys.stdout.write("\r" + " " * 60 + "\r")
    sys.stdout.flush()
    if not all_long:
        raise RuntimeError(f"No data returned for any country in {dataset_id}")
    return pd.concat(all_long, ignore_index=True)


def wide_monthly(
    long_df: pd.DataFrame,
    time_col: str,
    geo_col: str,
    value_col: str,
    countries: list = COUNTRIES,
    monthly_idx: pd.DatetimeIndex = MONTHLY_IDX,
    ffill_limit: int = 3,
) -> pd.DataFrame:
    """Pivot long → wide and reindex to the canonical monthly timeline."""
    df = long_df[[time_col, geo_col, value_col]].copy()
    df[geo_col] = df[geo_col].str.upper()

    def parse_time(t):
        t = str(t).strip()
        if len(t) == 4:   return pd.Timestamp(f"{t}-01-01")
        if "Q" in t:      return pd.Period(t, freq="Q").to_timestamp()
        return pd.to_datetime(t)

    df["date"] = df[time_col].apply(parse_time)
    df = df.pivot_table(index="date", columns=geo_col,
                        values=value_col, aggfunc="mean")
    for c in countries:
        if c not in df.columns:
            df[c] = np.nan
    df = df[countries]
    return df.reindex(monthly_idx).ffill(limit=ffill_limit)


def discover(dataset_id: str, geo: str = "DE") -> dict:
    """Debug helper: print all valid dimension codes for a dataset."""
    url = f"{ESTAT_BASE}/{dataset_id}"
    r = requests.get(
        url,
        params=[("geo", geo), ("lastTimePeriod", "1"), ("format", "JSON"), ("lang", "EN")],
        timeout=60,
    )
    js = r.json()
    if "error" in js:
        raise RuntimeError(f"Discovery failed for {dataset_id}: {js['error']}")
    dims = js.get("dimension", {})
    print(f"\n=== {dataset_id} — dimension discovery ===")
    for dim in js.get("id", []):
        codes = list(dims[dim]["category"]["index"].keys())
        print(f"  {dim:20s}: {codes[:15]}{'...' if len(codes) > 15 else ''}")
    print("=" * (len(dataset_id) + 27))
    return dims


print("Utility functions loaded.")


## Main Data Pull

Pulls monthly SITC-level intra-EU export flows for all EU27 reporter/partner pairs from the Eurostat Comext API (DS-059331).

**Note on Greece:** The Comext API uses `GR` for Greece. `COMEXT_COUNTRIES` (with `GR`) is used exclusively here. A rename step at the end of `main()` converts all `GR` values to `EL` so every downstream file is consistent.

Output: `main_data/eu_intra_trade_sitc.csv`


In [ ]:
# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress() -> set:
    """Return set of already-completed 'REPORTER|YYYY-MM' keys."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return set(json.load(f))
    return set()


def save_progress(done: set) -> None:
    with open(PROGRESS_FILE, "w") as f:
        json.dump(list(done), f)


# ── Fetch helpers ─────────────────────────────────────────────────────────────
def fetch_one(reporter: str, month_str: str, product_code: str) -> pd.DataFrame | None:
    """
    Fetch a single (reporter, month, product_code) slice from the Comext API.
    Uses COMEXT_COUNTRIES (with GR) as the partner list.
    Returns DataFrame[reporter, partner, time, value] or None on failure.
    """
    params = {
        "freq":       "M",
        "reporter":   reporter,
        "partner":    COMEXT_COUNTRIES,
        "product":    product_code,
        "flow":       FLOW,
        "indicators": INDICATOR,
        "time":       month_str,
        "lang":       "en",
        "format":     "JSON",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(COMEXT_BASE, params=params, timeout=45)
            if r.status_code == 400:   # future date or no data
                return None
            if r.status_code != 200:
                print(f"    HTTP {r.status_code} (attempt {attempt}/{MAX_RETRIES})")
                time.sleep(2 ** attempt)
                continue

            data    = r.json()
            dim_ids = data.get("id", [])
            if not dim_ids:
                return None

            label_map = {
                d: list(data["dimension"][d]["category"]["index"].keys())
                for d in dim_ids
            }
            values = data.get("value", {})
            combos = list(iterproduct(*[range(len(label_map[d])) for d in dim_ids]))

            rows = []
            for i, combo in enumerate(combos):
                val = values.get(str(i))
                if val is None:
                    continue
                row = {dim_ids[j]: label_map[dim_ids[j]][combo[j]] for j in range(len(dim_ids))}
                row["value"] = float(val)
                rows.append(row)

            if not rows:
                return None

            df = pd.DataFrame(rows)
            df = df[df["reporter"] != df["partner"]].copy()
            return df[["time", "reporter", "partner", "value"]]

        except Exception as e:
            print(f"    Exception on attempt {attempt}: {e}")
            time.sleep(2 ** attempt)

    return None


def fetch_product_group(reporter: str, month_str: str, label: str, codes: list) -> pd.Series | None:
    """Fetch and sum all SITC codes for one product group; returns Series keyed by (reporter, partner)."""
    frames = []
    for code in codes:
        df = fetch_one(reporter, month_str, code)
        time.sleep(SLEEP_SEC)
        if df is not None:
            frames.append(df.set_index(["reporter", "partner"])["value"])

    if not frames:
        return None

    combined      = pd.concat(frames, axis=1).sum(axis=1)
    combined.name = label
    return combined


# ── Main loop ─────────────────────────────────────────────────────────────────
def main():
    done   = load_progress()
    months = build_months(START_MONTH, END_MONTH)

    all_pairs  = [(r, m) for r in COMEXT_COUNTRIES for m in months]
    todo_pairs = [(r, m) for r, m in all_pairs if f"{r}|{m}" not in done]
    total      = len(all_pairs)
    completed  = total - len(todo_pairs)

    print(f"Total work:   {total} (reporter, month) pairs")
    print(f"Already done: {completed}  |  Remaining: {len(todo_pairs)}")
    if not todo_pairs:
        print("Nothing left to do — dataset is complete!")
    else:
        print(f"Resuming from reporter={todo_pairs[0][0]}, month={todo_pairs[0][1]}\n")

    for i, (reporter, month_str) in enumerate(todo_pairs, start=1):
        key = f"{reporter}|{month_str}"
        pct = (completed + i - 1) / total * 100
        print(f"[{completed+i}/{total} | {pct:.1f}%] {reporter} {month_str} ...", end=" ", flush=True)

        group_series = {}
        for label, codes in PRODUCT_GROUPS.items():
            s = fetch_product_group(reporter, month_str, label, codes)
            if s is not None:
                group_series[label] = s

        if not group_series:
            print("no data.")
            done.add(key)
            save_progress(done)
            continue

        df_wide = pd.concat(group_series, axis=1).reset_index()
        df_wide.insert(0, "time", month_str)
        for label in PRODUCT_GROUPS:
            if label not in df_wide.columns:
                df_wide[label] = 0.0
        col_order = ["time", "reporter", "partner"] + list(PRODUCT_GROUPS.keys())
        df_wide   = df_wide[col_order]

        write_header = not RAW_TRADE_CSV.exists()
        df_wide.to_csv(RAW_TRADE_CSV, mode="a", index=False, header=write_header)

        print(f"{len(df_wide)} pairs written.")
        done.add(key)
        save_progress(done)

    # ── Rename GR → EL so all downstream files use the canonical Eurostat code ──
    if RAW_TRADE_CSV.exists():
        df_out = pd.read_csv(RAW_TRADE_CSV)
        df_out["reporter"] = df_out["reporter"].replace("GR", "EL")
        df_out["partner"]  = df_out["partner"].replace("GR", "EL")
        df_out.to_csv(RAW_TRADE_CSV, index=False)
        print("Greece code normalised: GR → EL in", RAW_TRADE_CSV)

    print(f"\nDone. Output → {RAW_TRADE_CSV}")


# ── Outer resume guard — skip entirely if the CSV is already complete ─────────
if already_done("eu_intra_trade_sitc.csv", MAIN_DATA_DIR):
    print(f"SKIP — {RAW_TRADE_CSV} already complete.")
else:
    main()


### Imputation for Main Trade Flows

Applies a hierarchical share-based imputation to missing product-category breakdowns in `eu_intra_trade_sitc.csv`.

Fallback levels (in order):
1. Pair-specific expanding-window history
2. Reporter-level expanding-window history
3. Global expanding-window history
4. Global full-dataset mean (last resort; leaks future — flags level 4 usage in log)

Output: `main_data/eu_intra_trade_sitc_imputed.csv`


In [ ]:
if already_done("eu_intra_trade_sitc_imputed.csv", MAIN_DATA_DIR):
    print(f"SKIP — {IMPUTED_TRADE_CSV} already exists.")
else:
    # ── Load raw data ─────────────────────────────────────────────────────────
    df = pd.read_csv(RAW_TRADE_CSV, parse_dates=["time"])
    df = df.sort_values(["reporter", "partner", "time"]).reset_index(drop=True)

    complete_mask = df[CATS].notna().all(axis=1)
    df_complete   = df[complete_mask].copy()

    df_complete["cat_total"] = df_complete[CATS].sum(axis=1)
    for cat in CATS:
        df_complete[f"{cat}_share"] = df_complete[cat] / df_complete["cat_total"]

    share_cols = [f"{cat}_share" for cat in CATS]

    print(f"Complete rows (all CATS present): {complete_mask.sum():,} / {len(df):,}")

    # ── Pair share lookup table ───────────────────────────────────────────────
    pair_share_history = {}
    for (rep, part), grp in df_complete.groupby(["reporter", "partner"]):
        pair_share_history[(rep, part)] = (
            grp.sort_values("time")[["time"] + share_cols].reset_index(drop=True)
        )
    print(f"Pairs with at least one complete row: {len(pair_share_history):,}")

    # ── Hierarchical share lookup ─────────────────────────────────────────────
    fallback_log = []
    rename_map   = {f"{c}_share": c for c in CATS}

    def get_expanding_shares(reporter, partner, before_time):
        key = (reporter, partner)
        if key in pair_share_history:
            past = pair_share_history[key]
            past = past[past["time"] < before_time]
            if len(past) >= MIN_HISTORY:
                return past[share_cols].mean().rename(rename_map), 1

        reporter_past = df_complete[
            (df_complete["reporter"] == reporter) & (df_complete["time"] < before_time)
        ]
        if len(reporter_past) >= MIN_HISTORY:
            return reporter_past[share_cols].mean().rename(rename_map), 2

        global_past = df_complete[df_complete["time"] < before_time]
        if len(global_past) >= 1:
            return global_past[share_cols].mean().rename(rename_map), 3

        return df_complete[share_cols].mean().rename(rename_map), 4

    # ── Row-level imputation ──────────────────────────────────────────────────
    def impute_row(row, shares):
        known_cats   = [c for c in CATS if not pd.isna(row[c])]
        unknown_cats = [c for c in CATS if pd.isna(row[c])]
        known_total  = row[known_cats].sum() if known_cats else 0.0
        unknown_total = row["TOTAL"] - known_total
        imputed       = row.copy()

        zero_cats    = [c for c in unknown_cats if shares[c] < ZERO_THRESHOLD]
        nonzero_cats = [c for c in unknown_cats if shares[c] >= ZERO_THRESHOLD]

        for cat in zero_cats:
            imputed[cat] = 0.0

        if nonzero_cats:
            nz_shares = shares[nonzero_cats]
            nz_shares = nz_shares / nz_shares.sum()
            for cat in nonzero_cats:
                imputed[cat] = unknown_total * nz_shares[cat]
        else:
            if known_cats and unknown_total > 0:
                largest_known = max(known_cats, key=lambda c: imputed[c])
                imputed[largest_known] += unknown_total

        return imputed

    # ── Apply ─────────────────────────────────────────────────────────────────
    df_clean = df.copy()
    nan_mask = df_clean[CATS].isna().any(axis=1)
    print(f"Rows to impute: {nan_mask.sum():,}")

    for idx_row, row in df_clean[nan_mask].iterrows():
        shares, level = get_expanding_shares(row["reporter"], row["partner"], row["time"])
        df_clean.loc[idx_row] = impute_row(row, shares)
        fallback_log.append({"reporter": row["reporter"], "partner": row["partner"],
                              "time": row["time"], "level_used": level})

    df_clean[CATS] = df_clean[CATS].clip(lower=0)

    log_df = pd.DataFrame(fallback_log)
    if len(log_df):
        print("Fallback level usage:")
        print(log_df["level_used"].value_counts().sort_index())
        if (log_df["level_used"] == 4).any():
            print("  WARNING: level 4 (full-dataset mean) used — check for very early timestamps.")

    # ── Post-imputation integrity check ───────────────────────────────────────
    df_final = df_clean.sort_values(["reporter", "partner", "time"]).reset_index(drop=True)
    discrepancy = (df_final[CATS].sum(axis=1) - df_final["TOTAL"]).abs()
    print(f"Max category-sum vs TOTAL discrepancy: {discrepancy.max():.2f}")
    print(f"Rows where discrepancy > 1 EUR:        {(discrepancy > 1).sum()}")

    # ── Save ──────────────────────────────────────────────────────────────────
    df_final.to_csv(IMPUTED_TRADE_CSV, index=False)
    print(f"\nSaved: {IMPUTED_TRADE_CSV}")
    print(f"Shape: {df_final.shape}")
    print(f"NaN in CATS:  {df_final[CATS].isna().sum().sum()}")
    print(f"NaN in TOTAL: {df_final['TOTAL'].isna().sum()}")


# Node Feature Extraction Pipeline
**Intra-EU Trade Flow Prediction**

Extracts and saves all node-level features as monthly CSV files with shape `(T_months × 27_countries)`.

| Feature | Dataset | Output file(s) |
|---|---|---|
| GDP per capita | `namq_10_pc` | `gdp_per_capita_monthly.csv` |
| Economic Sentiment Indicator | `ei_bssi_m_r2` | `esi_monthly.csv` |
| HICP (4 special aggregates) | `prc_hicp_midx` | `hicp_{agg}_monthly.csv` |
| Producer Prices (28 NACE divisions) | `sts_inpp_m` | `ppi_{nace}_monthly.csv` |
| Multilateral resistance | computed from trade CSV | `multilateral_resistance.csv` |
| Bond yields (raw + spread) | `irt_lt_mcby_m` | `bond_yield_raw_monthly.csv`, `bond_yield_spread_monthly.csv` |


## GDP per Capita
`namq_10_pc` | Quarterly current prices EUR/inhabitant | B1GQ | NSA

Quarterly data is forward-filled into monthly frequency. Fetched from `2013-Q4` so the first quarterly lag correctly covers `2014-01`. Each month receives the value of the most recently completed quarter (`ffill_limit=3`).


In [ ]:
if already_done("gdp_per_capita_monthly.csv"):
    print("SKIP — gdp_per_capita_monthly.csv already exists.")
else:
    js_gdp = eurostat_json("namq_10_pc", {
        "unit":            "CP_EUR_HAB",
        "na_item":         "B1GQ",
        "s_adj":           "NSA",
        "geo":             COUNTRIES,
        "sinceTimePeriod": "2013-Q4",
        "untilTimePeriod": "2025-Q3",
        "format":          "JSON",
        "lang":            "EN",
    })

    time_col = [c for c in js_gdp.columns if "time" in c.lower()][0]
    geo_col  = [c for c in js_gdp.columns if "geo"  in c.lower()][0]
    wide_gdp = wide_monthly(js_gdp, time_col, geo_col, "value", ffill_limit=3)
    save(wide_gdp, "gdp_per_capita_monthly.csv")
    wide_gdp.head()


## Economic Sentiment Indicator (ESI)
`ei_bssi_m_r2` | Monthly | Seasonally adjusted | Long-run mean = 100, SD = 10

In [ ]:
if already_done("esi_monthly.csv"):
    print("SKIP — esi_monthly.csv already exists.")
else:
    js_esi = eurostat_json("ei_bssi_m_r2", {
        "indic":           "BS-ESI-I",
        "s_adj":           "SA",
        "geo":             COUNTRIES,
        "sinceTimePeriod": START_MONTH,
        "untilTimePeriod": END_MONTH,
        "format":          "JSON",
        "lang":            "EN",
    })
    time_col = [c for c in js_esi.columns if "time" in c.lower()][0]
    geo_col  = [c for c in js_esi.columns if "geo"  in c.lower()][0]
    wide_esi = wide_monthly(js_esi, time_col, geo_col, "value")
    save(wide_esi, "esi_monthly.csv")
    wide_esi.head()


## HICP — Special Aggregates (Index, 2015=100)
`prc_hicp_midx` | Unit: `I15` | Not seasonally adjusted

Four special aggregates selected:

| Code | Label |
|---|---|
| `FOOD` | Food including non-alcoholic beverages |
| `NRG` | Energy |
| `IGD_NNRG` | Non-energy industrial goods |
| `TOT_X_NRG_FOOD` | All items excluding energy and food (services + NEIG) |

In [ ]:
for coicop, out_name in HICP_AGGREGATES.items():
    if already_done(out_name):
        print(f"  SKIP  {out_name} already exists.")
        continue

    print(f"\nFetching HICP coicop={coicop} → {out_name}")
    js_h = eurostat_json("prc_hicp_midx", {
        "unit":            HICP_UNIT,
        "coicop":          coicop,
        "geo":             COUNTRIES,
        "sinceTimePeriod": START_MONTH,
        "untilTimePeriod": END_MONTH,
        "format":          "JSON",
        "lang":            "EN",
    })
    time_col = [c for c in js_h.columns if "time" in c.lower()][0]
    geo_col  = [c for c in js_h.columns if "geo"  in c.lower()][0]
    wide_h   = wide_monthly(js_h, time_col, geo_col, "value")
    save(wide_h, out_name)

print("\nAll HICP aggregates done.")


## Producer Prices in Industry (PPI)
`sts_inpp_m` | Unit: `I21` (Index, 2021=100) | NSA | Total market

NACE divisions, one CSV per division. Organised by trade category for readability.

| Trade category | NACE divisions |
|---|---|
| Raw_materials | B08 |
| Energy | C19, D35 |
| Food_drinks_tobacco | C10, C11, C12 |
| Chemicals | C20, C21, C22, C23 |
| Machinery_vehicles | C26, C27, C28, C29, C30 |
| Other_manufactured | C13, C14, C15, C16, C17, C18, C24, C25, C31, C32 |

In [ ]:
for category, nace_list in PPI_DIVISIONS.items():
    print(f"\n{'─'*55}")
    print(f"  Trade category: {category}")
    print(f"{'─'*55}")

    for nace in nace_list:
        out_name = f"ppi_{nace.lower()}_monthly.csv"
        if already_done(out_name):
            print(f"  SKIP  {out_name} already exists.")
            continue

        print(f"  Fetching nace_r2={nace} ...")
        try:
            js_ppi = eurostat_json("sts_inpp_m", {
                "unit":            PPI_UNIT,
                "indic_bt":        PPI_INDIC,
                "nace_r2":         nace,
                "geo":             COUNTRIES,
                "sinceTimePeriod": START_MONTH,
                "untilTimePeriod": END_MONTH,
                "format":          "JSON",
                "lang":            "EN",
            })
            time_col = [c for c in js_ppi.columns if "time" in c.lower()][0]
            geo_col  = [c for c in js_ppi.columns if "geo"  in c.lower()][0]
            wide_ppi = wide_monthly(js_ppi, time_col, geo_col, "value")
            save(wide_ppi, out_name)
        except RuntimeError as exc:
            print(f"  ERROR for {nace}: {exc} — skipping.")

print("\nAll PPI divisions done.")


## PPI Imputation

Some countries have very limited or almost no industrial activity, which is reflected in their PPI data as consistently missing or near-zero values. Rather than using a hardcoded list, we identify these countries directly from the data: if a country has very few reported values or only trivial values, it is treated as a small/limited economy.

**Imputation strategy:**
- For each month, if a country is identified as small/limited (based on data coverage or value), missing values are filled with 0. This is a correct signal for lack of industrial activity.
- For countries that have previously reported non-trivial PPI values, missing values are imputed using the mean of their own historical data (up to that month).
- If a country has never reported a non-trivial value, missing values are also filled with 0.

This approach ensures that imputation is data-driven and only imputes based on historical scale for countries where it makes sense. All files are overwritten in place. A sentinel file (`ppi_imputation.done`) is written when complete so this cell is safely skippable on re-runs.


In [ ]:
# PPI Imputation: Data-driven strategy for small/limited economies
SENTINEL = NODE_DIR / "ppi_imputation.done"

if SENTINEL.exists():
    print("SKIP — PPI imputation already done.")
else:
    ppi_files = sorted(NODE_DIR.glob("ppi_*_monthly.csv"))

    if not ppi_files:
        print("WARNING: No PPI files found in node_features/ — run PPI pull first.")
    else:
        total_filled = 0
        for fpath in ppi_files:
            df = pd.read_csv(fpath, index_col=0, parse_dates=True)
            filled_this_file = 0

            # Identify small/limited activity countries based on data
            # Criteria: country has <10% non-NaN values or max value < 1.0
            small_countries = set()
            for country in df.columns:
                non_nan_ratio = df[country].notna().sum() / len(df)
                max_val = df[country].max(skipna=True)
                if (non_nan_ratio < 0.1) or (pd.isna(max_val) or max_val < 1.0):
                    small_countries.add(country)

            # For all other countries, check if they have ever reported a non-trivial value
            nontrivial_countries = set(
                c for c in df.columns if (df[c].notna().sum() > 0 and df[c].max(skipna=True) >= 1.0)
            )

            for ts in df.index:
                row = df.loc[ts]
                nan_idx = row.index[row.isna()]
                if len(nan_idx) == 0:
                    continue
                for country in nan_idx:
                    if country in small_countries:
                        # Fill with 0 for small/limited activity economies
                        df.loc[ts, country] = 0.0
                        filled_this_file += 1
                    elif country in nontrivial_countries:
                        # Impute using that country's own historical mean (excluding current NaN)
                        hist = df.loc[:ts, country].dropna()
                        if len(hist) > 0:
                            df.loc[ts, country] = hist.mean()
                            filled_this_file += 1
                        else:
                            df.loc[ts, country] = 0.0
                            filled_this_file += 1
                    else:
                        # If neither, fallback to 0
                        df.loc[ts, country] = 0.0
                        filled_this_file += 1

            df.to_csv(fpath)   # overwrite original
            nans_after = df.isna().sum().sum()
            print(f"  {fpath.name:<40}  filled={filled_this_file:4d}  NaNs_remaining={nans_after}")
            total_filled += filled_this_file

        print(f"\nPPI imputation complete — {total_filled} values filled across {len(ppi_files)} files.")
        SENTINEL.touch()


## Multilateral Resistance

Computed from the imputed trade CSV (no API). Proxy: inverse of each country's mean log export volume across all partners for each period.

Output: `node_features/multilateral_resistance.csv`


In [ ]:
if already_done("multilateral_resistance.csv", NODE_DIR):
    print("SKIP — multilateral_resistance.csv already exists.")
elif not IMPUTED_TRADE_CSV.exists():
    print(f"WARNING: '{IMPUTED_TRADE_CSV}' not found — run the imputation cell first.")
else:
    df_main = pd.read_csv(IMPUTED_TRADE_CSV, parse_dates=["time"])
    df_main = df_main.sort_values(["time", "reporter", "partner"])

    # Build country index from canonical COUNTRIES list (already uses EL)
    c_idx   = {c: i for i, c in enumerate(COUNTRIES)}
    periods = sorted(df_main["time"].unique())

    A = np.zeros((len(periods), len(COUNTRIES), len(COUNTRIES)))
    for t, period in enumerate(periods):
        sl = df_main[df_main["time"] == period]
        for _, row in sl.iterrows():
            ri = c_idx.get(row["reporter"])
            pi = c_idx.get(row["partner"])
            if ri is not None and pi is not None:
                A[t, ri, pi] = row["TOTAL"]

    # Check for missing countries in the trade data
    trade_countries = set(df_main["reporter"]) | set(df_main["partner"])
    missing = [c for c in COUNTRIES if c not in trade_countries]
    if missing:
        print(f"WARNING: {missing} not found in trade data — their MR will be NaN.")

    A_log               = np.log1p(A)
    mean_trade_vol      = A_log.mean(axis=2)
    multilateral_resist = 1.0 / (mean_trade_vol + 1e-8)

    mr_df = pd.DataFrame(
        multilateral_resist,
        index   = pd.DatetimeIndex(periods),
        columns = COUNTRIES,
    )
    mr_df = mr_df.reindex(MONTHLY_IDX).ffill(limit=3)
    save(mr_df, "multilateral_resistance.csv", NODE_DIR)


## EMU Convergence Criterion Bond Yields (10Y Government Bonds)
`irt_lt_mcby_m` | Monthly | Percent per annum

Two output files:
1. `bond_yield_raw_monthly.csv` — raw 10Y yield per country
2. `bond_yield_spread_monthly.csv` — spread over the German Bund (country yield − DE yield)

**Estonia (EE)** has never issued 10Y government bonds and has structural NaNs throughout. Imputed using the EU average yield for each month.


In [ ]:
if already_done("bond_yield_raw_monthly.csv") and already_done("bond_yield_spread_monthly.csv"):
    print("SKIP — both bond yield files already exist.")
else:
    js_bond = eurostat_json("irt_lt_mcby_m", {
        "int_rt":          "MCBY",
        "geo":             COUNTRIES,
        "sinceTimePeriod": START_MONTH,
        "untilTimePeriod": END_MONTH,
        "format":          "JSON",
        "lang":            "EN",
    })

    time_col = [c for c in js_bond.columns if "time" in c.lower()][0]
    geo_col  = [c for c in js_bond.columns if "geo"  in c.lower()][0]

    wide_bond = wide_monthly(js_bond, time_col, geo_col, "value", ffill_limit=1)

    # ── Impute Estonia (EE) — structural NaN (no 10Y bond issuance) ───────────
    ee_nan_count = wide_bond["EE"].isna().sum()
    if ee_nan_count > 0:
        eu_mean = wide_bond.drop(columns=["EE"]).mean(axis=1)
        wide_bond["EE"] = wide_bond["EE"].fillna(eu_mean)
        print(f"Estonia (EE) imputed with EU average yield for {ee_nan_count} months.")

    # ── Spread over Bund (DE) ─────────────────────────────────────────────────
    if "DE" not in wide_bond.columns or wide_bond["DE"].isna().all():
        raise RuntimeError("German Bund (DE) series missing — cannot compute spreads.")

    bund        = wide_bond["DE"]
    wide_spread = wide_bond.subtract(bund, axis=0)   # country_i − DE for every month

    print(f"\nRaw yield sample (first 3 rows):")
    print(wide_bond.head(3))
    print(f"\nSpread over Bund sample (first 3 rows):")
    print(wide_spread.head(3))

    save(wide_bond,   "bond_yield_raw_monthly.csv",    NODE_DIR)
    save(wide_spread, "bond_yield_spread_monthly.csv",  NODE_DIR)


# Edge Features Extraction Pipeline

Bilateral structural features from CEPII GeoDist, expanded to a monthly timeseries.

**Before running:** download `dist_cepii.xls` from http://www.cepii.fr/distance/dist_cepii.zip and place it in the notebook directory (or update `GEODIST_FILE` in Cell B).

Output: `edge_features/eu27_bilateral_static_features.csv`
Columns: `time, reporter, partner, comlang_ethno, contig, smctry, dist, distcap, distw, distwces`

Features are time-invariant — each directed pair is repeated identically across all months, producing one row per (time, reporter, partner).


In [ ]:
# ── CEPII-specific helpers (used only in this section) ───────────────────────
def load_geodist(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in (".xls", ".xlsx"):
        df = pd.read_excel(path, dtype={"iso_o": str, "iso_d": str})
    else:
        df = pd.read_csv(path, dtype={"iso_o": str, "iso_d": str})
    df.columns = df.columns.str.strip().str.lower()
    print(f"Loaded GeoDist: {len(df)} rows")
    print(f"Available columns: {list(df.columns)}\n")
    return df


def extract_eu27_features(df: pd.DataFrame) -> pd.DataFrame:
    eu27_iso3   = set(ISO3_TO_EUROSTAT.keys())
    missing_cols = [c for c in CEPII_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Columns not found in GeoDist file: {missing_cols}\nAvailable: {list(df.columns)}")

    mask  = df["iso_o"].isin(eu27_iso3) & df["iso_d"].isin(eu27_iso3)
    df_eu = df.loc[mask, ["iso_o", "iso_d"] + CEPII_COLS].copy()
    print(f"EU27 undirected pairs found in GeoDist: {len(df_eu)}")

    df_eu["reporter"] = df_eu["iso_o"].map(ISO3_TO_EUROSTAT)
    df_eu["partner"]  = df_eu["iso_d"].map(ISO3_TO_EUROSTAT)

    unmapped = df_eu[df_eu["reporter"].isna() | df_eu["partner"].isna()]
    if not unmapped.empty:
        print(f"Warning: {len(unmapped)} rows with unmapped ISO3 codes dropped.")
    df_eu = df_eu.dropna(subset=["reporter", "partner"])

    df_fwd = df_eu[["reporter", "partner"] + CEPII_COLS].copy()
    df_rev = df_eu[["reporter", "partner"] + CEPII_COLS].rename(
        columns={"reporter": "partner", "partner": "reporter"}
    )
    df_directed = (
        pd.concat([df_fwd, df_rev], ignore_index=True)
        .pipe(lambda d: d[d["reporter"] != d["partner"]])
        .drop_duplicates(subset=["reporter", "partner"])
        .sort_values(["reporter", "partner"])
        .reset_index(drop=True)
    )
    for col in ["comlang_ethno", "contig", "smctry"]:
        df_directed[col] = df_directed[col].astype("Int64")

    print(f"Directed pairs after expansion: {len(df_directed)}")
    return df_directed


def expand_to_timeseries(df_pairs: pd.DataFrame, months: list) -> pd.DataFrame:
    """Cross-join each directed pair with every month — features are constant across time."""
    months_df = pd.DataFrame({"time": months, "_key": 1})
    df_pairs  = df_pairs.copy()
    df_pairs["_key"] = 1
    df_ts = (
        pd.merge(months_df, df_pairs, on="_key")
        .drop(columns="_key")
        [["time", "reporter", "partner"] + CEPII_COLS]
        .sort_values(["time", "reporter", "partner"])
        .reset_index(drop=True)
    )
    return df_ts


# ── Main ──────────────────────────────────────────────────────────────────────
if already_done("eu27_bilateral_static_features.csv", EDGE_DIR):
    print("SKIP — eu27_bilateral_static_features.csv already exists.")
elif not GEODIST_FILE.exists():
    print(f"WARNING: '{GEODIST_FILE}' not found — place dist_cepii.xls in the notebook directory.")
else:
    months   = build_months(START_MONTH, END_MONTH)
    df_raw   = load_geodist(GEODIST_FILE)
    df_pairs = extract_eu27_features(df_raw)
    df_ts    = expand_to_timeseries(df_pairs, months)
    save(df_ts, "eu27_bilateral_static_features.csv", EDGE_DIR)
    print(f"Total rows: {len(df_ts):,}")


# Graph Features

| Feature | Source | Type | Output |
|---|---|---|---|
| 3-Month EURIBOR | ECB Data Portal API | Graph | `graph_features/euribor3m_monthly.csv` |
| VSTOXX | Manual download | Graph | `graph_features/vstoxx_monthly.csv` |

Graph-level features (EURIBOR, VSTOXX) are broadcast identically to all 27 country columns.


## 3-Month EURIBOR (Graph-level feature)
**Source:** ECB Data Portal API | **Series:** `FM.M.U2.EUR.RT.MM.EURIBOR3MD_.HSTA`

Historical close, average of daily fixings through each month. Percent per annum. Broadcast to all 27 country columns.

Output: `graph_features/euribor3m_monthly.csv`


In [ ]:
if already_done("euribor3m_monthly.csv", GRAPH_DIR):
    print("SKIP — euribor3m_monthly.csv already exists.")
else:
    url_eur = f"{ECB_BASE}/{ECB_EURIBOR_SERIES}"
    r_eur   = requests.get(
        url_eur,
        params={"startPeriod": START_MONTH, "endPeriod": END_MONTH, "format": "csvdata"},
        timeout=60,
    )
    r_eur.raise_for_status()

    eur_df = pd.read_csv(StringIO(r_eur.text))
    print("Columns:", eur_df.columns.tolist())
    print(eur_df.head())

    # ECB CSV format: TIME_PERIOD and OBS_VALUE columns
    eur_series = (
        eur_df[["TIME_PERIOD", "OBS_VALUE"]]
        .copy()
        .rename(columns={"TIME_PERIOD": "date", "OBS_VALUE": "euribor3m"})
    )
    eur_series["date"]      = pd.to_datetime(eur_series["date"])
    eur_series["euribor3m"] = pd.to_numeric(eur_series["euribor3m"], errors="coerce")
    eur_series              = eur_series.set_index("date")["euribor3m"]

    print(f"\nObservations: {len(eur_series)}")
    print(f"Range: {eur_series.index.min().date()} → {eur_series.index.max().date()}")
    print(eur_series.head())

    wide_eur = broadcast_graph_feature(eur_series)
    save(wide_eur, "euribor3m_monthly.csv", GRAPH_DIR)
    wide_eur.head()


## VSTOXX — Euro Stoxx 50 Volatility Index (Graph-level feature)
**Source:** Manual download from `stoxx.com` → `VSTOXX50.txt`

Daily closing values are averaged to monthly frequency. Broadcast to all 27 country columns.

**Before running:** place `VSTOXX50.txt` in the notebook directory (or update `VSTOXX_FILE` in Cell B).

Output: `graph_features/vstoxx_monthly.csv`


In [ ]:
if already_done("vstoxx_monthly.csv", GRAPH_DIR):
    print("SKIP — vstoxx_monthly.csv already exists.")
elif not VSTOXX_FILE.exists():
    print(f"WARNING: {VSTOXX_FILE} not found — place the STOXX file in the notebook directory.")
else:
    # ── Read ──────────────────────────────────────────────────────────────────
    # The file is semicolon-separated with a Date and Indexvalue column.
    # Decimal separator is "." (not ","). dtype=str for safe initial parse.
    vstoxx_raw = pd.read_csv(
        VSTOXX_FILE,
        sep=";",
        header=0,
        decimal=".",
        encoding="utf-8",
        engine="python",
        na_values=["-", "n.a.", "N/A", ""],
        dtype=str,
    )
    vstoxx_raw.columns = vstoxx_raw.columns.str.strip()
    print("Columns:", vstoxx_raw.columns.tolist())
    print(vstoxx_raw.head())

    # ── Parse and filter ──────────────────────────────────────────────────────
    vdf = vstoxx_raw[["Date", "Indexvalue"]].copy()
    vdf["date"]   = pd.to_datetime(vdf["Date"], dayfirst=True, errors="coerce")
    vdf["vstoxx"] = pd.to_numeric(vdf["Indexvalue"], errors="coerce")
    vdf = vdf.dropna(subset=["date", "vstoxx"])
    vdf = vdf[
        (vdf["date"] >= pd.Timestamp(START_MONTH)) &
        (vdf["date"] <= pd.Timestamp(END_MONTH + "-31"))
    ]

    print(f"Daily observations in range: {len(vdf)}")
    print(f"Date range: {vdf['date'].min().date()} → {vdf['date'].max().date()}")

    # ── Monthly average ───────────────────────────────────────────────────────
    monthly_vstoxx = (
        vdf.set_index("date")["vstoxx"]
           .resample("MS")
           .mean()
           .rename("VSTOXX_monthly_avg")
    )

    missing_months = MONTHLY_IDX[~MONTHLY_IDX.isin(monthly_vstoxx.index)]
    if len(missing_months) > 0:
        print(f"WARNING: {len(missing_months)} months have no VSTOXX data: {missing_months}")

    print(f"\nMonthly averages ({len(monthly_vstoxx)} months):")
    print(monthly_vstoxx.head(8))

    wide_vstoxx = broadcast_graph_feature(monthly_vstoxx)
    save(wide_vstoxx, "vstoxx_monthly.csv", GRAPH_DIR)
    wide_vstoxx.head()


## Coverage Report
Shape, NaN count, and date range for all output files.

In [ ]:
for dir_label, scan_dir in [("node_features", NODE_DIR), ("graph_features", GRAPH_DIR)]:
    print(f"\n{'='*70}")
    print(f"  COVERAGE REPORT — {dir_label}/")
    print(f"{'='*70}")
    print(f"  {'File':<45} {'Shape':>12}  {'NaNs':>6}  {'NaN%':>6}")
    print(f"  {'-'*65}")

    csv_files = sorted(scan_dir.glob("*.csv"))
    if not csv_files:
        print(f"  No CSV files found in {dir_label}/")
    else:
        for f in csv_files:
            try:
                df   = pd.read_csv(f, index_col=0, parse_dates=True)
                nans = df.isna().sum().sum()
                pct  = 100 * nans / df.size if df.size > 0 else 0
                print(f"  {f.name:<45} {str(df.shape):>12}  {nans:>6}  {pct:>5.1f}%")
            except Exception as e:
                print(f"  {f.name:<45} ERROR: {e}")

    print(f"{'='*70}")

print("\nEdge features (row count only — not a simple wide CSV):")
ef = list(EDGE_DIR.glob("*.csv"))
for f in ef:
    try:
        n = sum(1 for _ in open(f)) - 1
        print(f"  {f.name}: {n:,} rows")
    except Exception as e:
        print(f"  {f.name}: ERROR {e}")


# GNN Assembly

Assemble all sources into tensors ready for graph neural network training.

## Target output shapes

| Tensor | Shape | Description |
|---|---|---|
| Node features | `(T, N, F_node)` | One row per (time, country) — GDP, ESI, HICP×4, PPI×28, MR, bond yield×2 |
| Edge features | `(T, E, F_edge)` | One row per (time, reporter, partner) — CEPII static + trade flows |
| Graph features | `(T, F_graph)` | One value per month — EURIBOR, VSTOXX |
| Target (adjacency) | `(T, N, N)` | TOTAL trade value per directed pair |

Where `T` = number of months, `N` = 27, `E` = 27×26 = 702 directed pairs.

## Assembly steps
1. Load all node feature CSVs from `node_features/`, align on canonical monthly index and country list, stack.
2. Load edge features from `edge_features/`, merge with imputed trade data.
3. Verify consistent time range across all sources.
4. Save tensors (format `.pt`).


In [ ]:
# GNN Assembly: Save node, edge, graph tensors independently, and also node_plus_graph
import torch
import glob

# --- Node features ---
node_dir = NODE_DIR
node_csvs = sorted(glob.glob(str(node_dir / "*.csv")))
node_features = []
node_feature_names = []
for f in node_csvs:
    df = pd.read_csv(f, index_col=0, parse_dates=True)
    node_features.append(df)
    node_feature_names.extend([f"{f.split('/')[-1].replace('_monthly.csv','').replace('.csv','')}:{col}" for col in df.columns])

# Align on index and columns (time, country)
common_idx = node_features[0].index
common_cols = node_features[0].columns
for df in node_features[1:]:
    common_idx = common_idx.intersection(df.index)
    common_cols = common_cols.intersection(df.columns)

node_features = [df.loc[common_idx, common_cols] for df in node_features]
node_stack = np.stack([df.values for df in node_features], axis=-1)  # shape (T, N, F_node)
node_tensor = torch.tensor(node_stack, dtype=torch.float32)

# --- Graph features ---
graph_dir = GRAPH_DIR
graph_csvs = sorted(glob.glob(str(graph_dir / "*.csv")))
graph_features = []
for f in graph_csvs:
    df = pd.read_csv(f, index_col=0, parse_dates=True)
    graph_features.append(df)
common_gidx = graph_features[0].index
for df in graph_features[1:]:
    common_gidx = common_gidx.intersection(df.index)
graph_features = [df.loc[common_gidx] for df in graph_features]
graph_stack = np.concatenate([df.values for df in graph_features], axis=1)  # shape (T, F_graph)
graph_tensor = torch.tensor(graph_stack, dtype=torch.float32)

# --- Edge features ---
edge_dir = EDGE_DIR
edge_csvs = sorted(glob.glob(str(edge_dir / "*.csv")))
edge_features = []
for f in edge_csvs:
    df = pd.read_csv(f)
    edge_features.append(df)
from functools import reduce

def merge_edge_features(edge_features):
    base = edge_features[0]
    for df in edge_features[1:]:
        base = pd.merge(base, df, on=["time", "reporter", "partner"], how="outer")
    return base

edge_merged = merge_edge_features(edge_features)
edge_merged = edge_merged.sort_values(["time", "reporter", "partner"]).reset_index(drop=True)
edge_times = sorted(edge_merged["time"].unique())
countries = list(common_cols)
T = len(edge_times)
N = len(countries)
E = N * (N - 1)
edge_feat_cols = [c for c in edge_merged.columns if c not in ["time", "reporter", "partner"]]
F_edge = len(edge_feat_cols)
edge_tensor = np.zeros((T, E, F_edge), dtype=np.float32)
edge_idx_map = {(rep, part): i for i, (rep, part) in enumerate([(r, p) for r in countries for p in countries if r != p])}
for t_idx, t in enumerate(edge_times):
    df_t = edge_merged[edge_merged["time"] == t]
    for _, row in df_t.iterrows():
        rep, part = row["reporter"], row["partner"]
        if (rep in countries) and (part in countries) and (rep != part):
            e_idx = edge_idx_map[(rep, part)]
            edge_tensor[t_idx, e_idx, :] = row[edge_feat_cols].values
edge_tensor = torch.tensor(edge_tensor, dtype=torch.float32)

# --- node_plus_graph tensor (broadcast graph features to all nodes) ---
F_node = node_tensor.shape[2]
F_graph = graph_tensor.shape[1]
graph_broadcast = graph_tensor[:, None, :].expand(-1, N, -1)
node_plus_graph_tensor = torch.cat([node_tensor, graph_broadcast], dim=2)

# --- Save tensors ---
torch.save(node_tensor, str(DATA_DIR / "node_features.pt"))
torch.save(edge_tensor, str(DATA_DIR / "edge_features.pt"))
torch.save(graph_tensor, str(DATA_DIR / "graph_features.pt"))
torch.save(node_plus_graph_tensor, str(DATA_DIR / "node_plus_graph_features.pt"))
print("Saved node, edge, graph, and node_plus_graph tensors in .pt format.")
